# Buddhist Texts Corpus - Cleaning and Creation Pipeline

This notebook processes extracted text to create a clean, organized corpus:

## Pipeline Steps:
1. Load extracted pages from filtered_text/
2. Clean text (remove English, URLs, normalize)
3. Classify language (Sinhala, Pali, Mixed)
4. Create multi-format corpus:
   - Individual page files (by language)
   - Book-level concatenations
   - Consolidated training files
   - CSV metadata
   - JSON manifest
5. Create train/val/test splits
6. Generate statistics and reports

## Output Structure:
```
data/corpus/
├── raw/by_language/{sinhala,pali,mixed}/category/book_name/page_XXX.txt
├── raw/by_book/category/book_name.txt
├── consolidated/{sinhala,pali,mixed,full}_corpus.txt
├── splits/{sinhala,pali,mixed}/{train,validation,test}.txt
├── structured/corpus_{pages,books}.csv
└── metadata/corpus_manifest.json
```

## 1. Setup and Installation

In [25]:
# Install required packages
!pip install -q tqdm pandas

In [26]:
# Import libraries
import os
import re
import json
import unicodedata
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from datetime import datetime
from collections import defaultdict, Counter

import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 2. Mount Drive and Configuration

In [27]:
# Mount Google Drive
drive.mount('/content/drive')

# Project directory
BASE_DIR = Path('/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/')

if BASE_DIR.exists():
    print(f"✓ Project directory found: {BASE_DIR}")
else:
    print(f"✗ Project directory not found: {BASE_DIR}")
    print("Please update BASE_DIR")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Project directory found: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent


In [28]:
# Directory paths
EXTRACTED_DIR = BASE_DIR / "data" / "extracted"
FILTERED_TEXT_DIR = EXTRACTED_DIR / "filtered_text"  # Input: content pages only
METADATA_FILE = BASE_DIR / "data" / "metadata" / "metadata.json"
PALI_LEXICON = BASE_DIR / "data" / "pali_lexicon" / "pali_lexicons_single.tsv"

# Output directories
CORPUS_DIR = BASE_DIR / "data" / "corpus"
RAW_CORPUS_DIR = CORPUS_DIR / "raw"
BY_LANGUAGE_DIR = RAW_CORPUS_DIR / "by_language"
BY_BOOK_DIR = RAW_CORPUS_DIR / "by_book"
CONSOLIDATED_DIR = CORPUS_DIR / "consolidated"
SPLITS_DIR = CORPUS_DIR / "splits"
STRUCTURED_DIR = CORPUS_DIR / "structured"
CORPUS_METADATA_DIR = CORPUS_DIR / "metadata"

# Create directories
for dir_path in [BY_LANGUAGE_DIR, BY_BOOK_DIR, CONSOLIDATED_DIR,
                 SPLITS_DIR, STRUCTURED_DIR, CORPUS_METADATA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Create language subdirectories
for lang in ['sinhala', 'pali', 'mixed']:
    (BY_LANGUAGE_DIR / lang).mkdir(exist_ok=True)
    (SPLITS_DIR / lang).mkdir(exist_ok=True)

print("✓ Directory structure created")
print(f"  Input: {FILTERED_TEXT_DIR}")
print(f"  Output: {CORPUS_DIR}")

✓ Directory structure created
  Input: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/extracted/filtered_text
  Output: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus


## 3. Configuration Parameters

In [29]:
# Language classification threshold
CLASSIFICATION_THRESHOLD = 0.7  # 0.0-1.0
# Lower = more lenient (easier to classify as Sinhala/Pali)
# Higher = more strict (more will be mixed)

# Cleaning parameters
ENGLISH_THRESHOLD = 0.2  # Remove page if >20% English
MIN_PAGE_LENGTH = 50     # Minimum characters after cleaning

# Train/val/test split ratios
SPLIT_RATIOS = {
    'train': 0.8,
    'validation': 0.1,
    'test': 0.1
}

print("Configuration:")
print(f"  Language threshold: {CLASSIFICATION_THRESHOLD}")
print(f"  English threshold: {ENGLISH_THRESHOLD}")
print(f"  Min page length: {MIN_PAGE_LENGTH}")
print(f"  Split ratios: {SPLIT_RATIOS}")

Configuration:
  Language threshold: 0.7
  English threshold: 0.2
  Min page length: 50
  Split ratios: {'train': 0.8, 'validation': 0.1, 'test': 0.1}


## 4. Helper Functions

In [30]:
def load_json(filepath: Path, default=None):
    """Load JSON file"""
    if filepath.exists():
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    return default if default is not None else {}

def save_json(data, filepath: Path):
    """Save JSON file"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

def sanitize_filename(filename: str) -> str:
    """Remove invalid characters from filename"""
    filename = re.sub(r'[<>:"/\\|?*]', '_', filename)
    if len(filename) > 200:
        filename = filename[:200]
    return filename.strip()

def calculate_sinhala_ratio(text: str) -> float:
    """Calculate ratio of Sinhala characters"""
    if not text:
        return 0.0
    sinhala_chars = sum(1 for c in text if '\u0D80' <= c <= '\u0DFF')
    return sinhala_chars / len(text)

def calculate_english_ratio(text: str) -> float:
    """Calculate ratio of English characters"""
    if not text:
        return 0.0
    english_chars = sum(1 for c in text if c.isascii() and c.isalpha())
    return english_chars / len(text)

print("✓ Helper functions loaded")

✓ Helper functions loaded


## 5. Text Cleaning Functions

In [31]:
# ============================================================================
# UPDATED TEXT CLEANING FUNCTIONS
# ============================================================================
# Replace the clean_text function in Section 5 with this version
# This removes English text while preserving Sinhala/Pali on the same page
# ============================================================================

def remove_english_from_line(line: str) -> str:
    """
    Remove English words/sentences from a line while preserving Sinhala/Pali.

    Strategy:
    1. Split into words
    2. Keep word if it contains Sinhala characters
    3. Remove word if it's primarily English
    4. Keep numbers and punctuation
    """

    if not line:
        return ""

    # Split into tokens (words and spaces)
    # This preserves spacing structure
    tokens = re.findall(r'\S+|\s+', line)

    cleaned_tokens = []
    for token in tokens:
        # If it's whitespace, keep it (for spacing)
        if token.strip() == '':
            cleaned_tokens.append(' ')  # Normalize to single space
            continue

        # Check if token contains Sinhala characters
        has_sinhala = any('\u0D80' <= c <= '\u0DFF' for c in token)

        # Count English letters vs total letters
        english_letters = sum(1 for c in token if c.isascii() and c.isalpha())
        total_letters = sum(1 for c in token if c.isalpha())

        # Decision logic
        if has_sinhala:
            # Keep anything with Sinhala characters
            cleaned_tokens.append(token)

        elif english_letters > 0 and total_letters > 0:
            # It's an English word

            # Exception 1: Very short words (1-2 chars) might be abbreviations
            if len(token) <= 2:
                cleaned_tokens.append(token)

            # Exception 2: Common Buddhist terms in English
            # (you can customize this list)
            elif token.lower() in ['buddha', 'dhamma', 'sangha', 'tripitaka']:
                cleaned_tokens.append(token)

            # Otherwise, remove the English word
            # (don't add to cleaned_tokens)

        else:
            # It's punctuation, numbers, or symbols - keep it
            cleaned_tokens.append(token)

    # Join tokens and clean up spacing
    result = ''.join(cleaned_tokens)
    result = re.sub(r' +', ' ', result)  # Multiple spaces to single

    return result.strip()


def clean_text(text: str) -> str:
    """
    Clean text by removing unwanted content.

    Removes:
    - URLs
    - Email addresses
    - English text (word-by-word, not entire pages)
    - Excessive whitespace
    - Control characters
    - Page numbers (isolated)

    Preserves:
    - All Sinhala text
    - Pali transliterations in Sinhala script
    - Sinhala punctuation
    - Numbers in context
    """

    if not text:
        return ""

    # Step 1: Remove URLs
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)

    # Step 2: Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # Step 3: Remove isolated page numbers (e.g., "142" on its own line)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # Step 4: Remove English text (line by line)
    # This is the KEY change - we process each line to remove English
    cleaned_lines = []
    for line in text.split('\n'):
        cleaned_line = remove_english_from_line(line)
        if cleaned_line.strip():  # Only keep non-empty lines
            cleaned_lines.append(cleaned_line)
    text = '\n'.join(cleaned_lines)

    # Step 5: Remove control characters (but keep line breaks)
    text = ''.join(char for char in text if char == '\n' or not unicodedata.category(char).startswith('C'))

    # Step 6: Normalize Unicode (NFC - canonical composition)
    text = unicodedata.normalize('NFC', text)

    # Step 7: Clean up whitespace
    # Replace multiple spaces with single space
    text = re.sub(r' +', ' ', text)
    # Replace multiple newlines with double newline
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)
    # Remove leading/trailing whitespace from lines
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(line for line in lines if line)

    # Step 8: Remove duplicate lines (common OCR artifact)
    seen_lines = set()
    unique_lines = []
    for line in text.split('\n'):
        if line not in seen_lines:
            seen_lines.add(line)
            unique_lines.append(line)
    text = '\n'.join(unique_lines)

    return text.strip()


def should_remove_page(text: str, min_length: int = 50) -> tuple:
    """
    Determine if page should be removed after cleaning.

    NOW: Only remove if page is too short after cleaning
    (We no longer remove based on English content since we remove English text)

    Returns:
        (should_remove: bool, reason: str)
    """

    if len(text.strip()) < min_length:
        return True, f"Too short after cleaning ({len(text)} chars)"

    # Check if there's any Sinhala content left
    sinhala_chars = sum(1 for c in text if '\u0D80' <= c <= '\u0DFF')
    if sinhala_chars < 20:
        return True, f"Insufficient Sinhala content ({sinhala_chars} chars)"

    return False, "OK"


print("✓ Updated text cleaning functions loaded")
print("  → English words will be removed while preserving Sinhala/Pali")
print("  → Pages kept if they have sufficient Sinhala content after cleaning")

✓ Updated text cleaning functions loaded
  → English words will be removed while preserving Sinhala/Pali
  → Pages kept if they have sufficient Sinhala content after cleaning


## 6. Sinhala/Pali Language Classifier

In [32]:
"""
Enhanced Production Classifier with Pali Lexicon Support
=========================================================

This classifier uses both Sinhala and Pali lexicons for accurate classification.

Author: Multi-Lingual Buddhist Conversational Agent Project
Date: November 2025
"""

import re
from typing import Tuple, Dict, Set, Optional
from pathlib import Path
import urllib.request
import pandas as pd


# ============================================
# LEXICON LOADING FUNCTIONS
# ============================================

def download_google_lexicon(save_path: str = "lexicon.tsv") -> str:
    """
    Download the full Google Sinhala lexicon from GitHub.
    """
    url = "https://raw.githubusercontent.com/google/language-resources/master/si/data/lexicon.tsv"

    print(f"Downloading Sinhala lexicon from {url}...")

    try:
        urllib.request.urlretrieve(url, save_path)
        print(f"✓ Lexicon downloaded to: {save_path}")
        return save_path
    except Exception as e:
        print(f"✗ Download failed: {e}")
        print("Please download manually from:")
        print(url)
        raise


def load_sinhala_lexicon_from_tsv(tsv_path: str) -> Set[str]:
    """
    Load Sinhala words from Google's TSV lexicon file.

    Format: word<TAB>phonemic_transcription
    """
    lexicon = set()

    with open(tsv_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith('#'):
                continue

            parts = line.split('\t')
            if len(parts) >= 1:
                word = parts[0].strip()
                if word:
                    lexicon.add(word)

    print(f"✓ Loaded {len(lexicon):,} Sinhala words")
    return lexicon


def load_pali_lexicon(tsv_path: str) -> Set[str]:
    """
    Load Pali words from your custom lexicon file.

    Supports:
    - TSV with 'word' column
    - Plain text (one word per line)

    Args:
        tsv_path: Path to pali_lexicons_single.tsv

    Returns:
        Set of Pali words
    """
    path = Path(tsv_path)

    if not path.exists():
        raise FileNotFoundError(f"Pali lexicon not found: {tsv_path}")

    print(f"Loading Pali lexicon from {path.name}...")

    try:
        # Try loading as TSV with pandas
        df = pd.read_csv(path, sep='\t', encoding='utf-8')

        # Check if 'word' column exists
        if 'word' in df.columns:
            lexicon = set(df['word'].dropna().astype(str).tolist())
        else:
            # Fallback: use first column
            lexicon = set(df.iloc[:, 0].dropna().astype(str).tolist())

    except Exception as e:
        # Fallback: Load as plain text (one word per line)
        print(f"⚠ Could not load as TSV, trying plain text format: {e}")
        lexicon = set()
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word and not word.startswith('#'):
                    lexicon.add(word)

    print(f"✓ Loaded {len(lexicon):,} Pali words")
    return lexicon


# ============================================
# ENHANCED CLASSIFIER WITH DUAL LEXICONS
# ============================================

class DualLexiconSinhalaPaliClassifier:
    """
    Production classifier using both Sinhala and Pali lexicons.
    """

    def __init__(
        self,
        sinhala_lexicon_path: Optional[str] = None,
        pali_lexicon_path: Optional[str] = None,
        auto_download_sinhala: bool = False
    ):
        """
        Initialize classifier with Sinhala and Pali lexicons.

        Args:
            sinhala_lexicon_path: Path to Sinhala lexicon TSV
            pali_lexicon_path: Path to Pali lexicon TSV (your file)
            auto_download_sinhala: Auto-download Sinhala lexicon if not found

        Examples:
            # Load both lexicons
            classifier = DualLexiconSinhalaPaliClassifier(
                sinhala_lexicon_path="data/sinhala_lexicon.tsv",
                pali_lexicon_path="data/pali_lexicon/pali_lexicons_single.tsv"
            )

            # Auto-download Sinhala, provide Pali
            classifier = DualLexiconSinhalaPaliClassifier(
                auto_download_sinhala=True,
                pali_lexicon_path="data/pali_lexicon/pali_lexicons_single.tsv"
            )
        """
        # Initialize patterns
        self._initialize_patterns()

        # Load Sinhala lexicon
        if sinhala_lexicon_path:
            self.sinhala_lexicon = load_sinhala_lexicon_from_tsv(sinhala_lexicon_path)
            self.sinhala_source = sinhala_lexicon_path
        elif auto_download_sinhala:
            try:
                lexicon_file = "sinhala_lexicon.tsv"
                download_google_lexicon(lexicon_file)
                self.sinhala_lexicon = load_sinhala_lexicon_from_tsv(lexicon_file)
                self.sinhala_source = f"auto-downloaded: {lexicon_file}"
            except Exception as e:
                print(f"⚠ Download failed, using embedded sample: {e}")
                self.sinhala_lexicon = set(self.sinhala_common_words)
                self.sinhala_source = "embedded_sample"
        else:
            self.sinhala_lexicon = set(self.sinhala_common_words)
            self.sinhala_source = "embedded_sample"

        # Load Pali lexicon
        if pali_lexicon_path:
            self.pali_lexicon = load_pali_lexicon(pali_lexicon_path)
            self.pali_source = pali_lexicon_path
        else:
            self.pali_lexicon = set(self.pali_common_words)
            self.pali_source = "embedded_sample"
            print("⚠ No Pali lexicon provided, using embedded sample only")

        print(f"\n{'='*60}")
        print("CLASSIFIER INITIALIZED")
        print(f"{'='*60}")
        print(f"Sinhala lexicon: {len(self.sinhala_lexicon):,} words")
        print(f"  Source: {self.sinhala_source}")
        print(f"Pali lexicon:    {len(self.pali_lexicon):,} words")
        print(f"  Source: {self.pali_source}")
        print(f"{'='*60}\n")

    def _initialize_patterns(self):
        """Initialize grammatical patterns"""
        # Pali morphological patterns
        self.pali_nominal_endings = [
            r'තෝ\b', r'ස්ස\b', r'ස්මා\b', r'ස්මිං\b', r'ම්හි\b',
            r'ානං\b', r'ේසං\b', r'ේහි\b', r'ේසු\b', r'ාය\b',
        ]

        self.pali_verbal_endings = [
            r'ති\b', r'න්ති\b', r'තේ\b', r'න්තේ\b',
            r'මානො\b', r'න්තො\b', r'ත්වා\b', r'ත්වාන\b',
        ]

        self.pali_particles = [
            r'\bච\b', r'\bව\b', r'\bපි\b', r'\bහි\b', r'\bඛො\b',
            r'\bතං\b', r'\bස\b', r'\bඑවං\b',
        ]

        # Sinhala morphological patterns
        self.sinhala_case_markers = [
            r'ට\b', r'ගේ\b', r'ගෙ\b', r'හි\b', r'යේ\b',
            r'ෙයන්\b', r'ෙහි\b', r'ෙන්\b', r'ින්\b',
        ]

        self.sinhala_verbal_markers = [
            r'\bවූ\b', r'\bවන\b', r'\bකළ\b', r'\bගත්\b', r'\bදුන්\b',
            r'මු\b', r'මි\b', r'හ\b', r'වා\b', r'නවා\b',
            r'න්න\b', r'මින්\b', r'ලා\b', r'යි\b', r'ද\b', r'නේ\b',
        ]

        # Embedded common words (fallback)
        self.sinhala_common_words = [
            'මා', 'මාගේ', 'මං', 'මම', 'අපි', 'අපේ', 'අපගේ',
            'ඔබ', 'ඔබේ', 'ඔබගේ', 'ඔයා', 'එයා', 'මේ', 'මෙය', 'ඒ', 'එය',
            'කොහෙද', 'මොකද', 'කවදා', 'එහෙම', 'මෙහෙම', 'නෑ', 'නැහැ',
        ]

        self.pali_common_words = [
            'භගවතෝ', 'තථාගතස්ස', 'සබ්බඤ්ඤුනො', 'සම්මාසම්බුද්ධස්ස',
            'ධම්මස්ස', 'සඞ්ඝස්ස', 'භික්ඛුනො', 'භික්ඛූනං',
            'නමො', 'භන්තේ', 'අරහං', 'බුද්ධො',
        ]

        # Compile patterns
        self.pali_nominal_pattern = re.compile('|'.join(self.pali_nominal_endings))
        self.pali_verbal_pattern = re.compile('|'.join(self.pali_verbal_endings))
        self.pali_particle_pattern = re.compile('|'.join(self.pali_particles))

        self.sinhala_case_pattern = re.compile('|'.join(self.sinhala_case_markers))
        self.sinhala_verbal_pattern = re.compile('|'.join(self.sinhala_verbal_markers))

    def count_pali_features(self, text: str) -> Dict[str, int]:
        """
        Count Pali linguistic features including lexicon matches.
        """
        words = text.split()

        features = {
            'nominal_endings': len(self.pali_nominal_pattern.findall(text)),
            'verbal_endings': len(self.pali_verbal_pattern.findall(text)),
            'particles': len(self.pali_particle_pattern.findall(text)),
            'common_words': sum(text.count(word) for word in self.pali_common_words),
            'long_compounds': len([w for w in words if len(w) > 30]),
            'lexicon_matches': sum(1 for word in words if word in self.pali_lexicon)  # NEW!
        }
        return features

    def count_sinhala_features(self, text: str) -> Dict[str, int]:
        """
        Count Sinhala linguistic features including lexicon matches.
        """
        words = text.split()

        features = {
            'case_markers': len(self.sinhala_case_pattern.findall(text)),
            'verbal_markers': len(self.sinhala_verbal_pattern.findall(text)),
            'common_words': sum(text.count(word) for word in self.sinhala_common_words),
            'lexicon_matches': sum(1 for word in words if word in self.sinhala_lexicon)  # ENHANCED!
        }
        return features

    def classify(self, text: str, threshold: float = 0.70) -> Tuple[str, Dict]:
        """
        Classify text using a unified confidence score combining all signals.

        Args:
            text: Input text to classify
            threshold: Minimum confidence for pure classification (default: 0.70)
                      Above threshold → 'sinhala' or 'pali'
                      Below threshold → 'mixed'

        Returns:
            (classification, debug_info) tuple
        """
        if not text or len(text.strip()) < 30:
            return 'mixed', {
                'reason': 'Text too short',
                'confidence': 0.0,
                'text_length': len(text)
            }

        # ============================================
        # STEP 1: Extract all features
        # ============================================

        pali_features = self.count_pali_features(text)
        sinhala_features = self.count_sinhala_features(text)

        # Feature totals (morphological patterns)
        pali_feature_total = sum(pali_features.values())
        sinhala_feature_total = sum(sinhala_features.values())

        # Lexicon matches
        words = text.split()
        word_count = len(words)

        pali_lexicon_matches = pali_features['lexicon_matches']
        sinhala_lexicon_matches = sinhala_features['lexicon_matches']

        # ============================================
        # STEP 2: Calculate normalized scores (0-1)
        # ============================================

        # A. Morphological feature score
        total_features = pali_feature_total + sinhala_feature_total
        if total_features > 0:
            pali_feature_score = pali_feature_total / total_features
            sinhala_feature_score = sinhala_feature_total / total_features
        else:
            pali_feature_score = 0.0
            sinhala_feature_score = 0.0

        # B. Lexicon coverage score
        if word_count > 0:
            pali_lexicon_score = pali_lexicon_matches / word_count
            sinhala_lexicon_score = sinhala_lexicon_matches / word_count
        else:
            pali_lexicon_score = 0.0
            sinhala_lexicon_score = 0.0

        # ============================================
        # STEP 3: Weighted combined confidence
        # ============================================

        # Weight lexicon more heavily (more reliable than patterns)
        LEXICON_WEIGHT = 0.7
        FEATURE_WEIGHT = 0.3

        pali_confidence = (
            pali_lexicon_score * LEXICON_WEIGHT +
            pali_feature_score * FEATURE_WEIGHT
        )

        sinhala_confidence = (
            sinhala_lexicon_score * LEXICON_WEIGHT +
            sinhala_feature_score * FEATURE_WEIGHT
        )

        # ============================================
        # STEP 4: Classify based on confidence
        # ============================================

        debug_info = {
            'text_length': len(text),
            'word_count': word_count,

            # Raw counts
            'pali_features': pali_features,
            'sinhala_features': sinhala_features,
            'pali_feature_total': pali_feature_total,
            'sinhala_feature_total': sinhala_feature_total,
            'pali_lexicon_matches': pali_lexicon_matches,
            'sinhala_lexicon_matches': sinhala_lexicon_matches,

            # Normalized scores
            'pali_feature_score': pali_feature_score,
            'sinhala_feature_score': sinhala_feature_score,
            'pali_lexicon_score': pali_lexicon_score,
            'sinhala_lexicon_score': sinhala_lexicon_score,

            # Final confidences
            'pali_confidence': pali_confidence,
            'sinhala_confidence': sinhala_confidence,

            # Metadata
            'threshold': threshold,
            'sinhala_lexicon_size': len(self.sinhala_lexicon),
            'pali_lexicon_size': len(self.pali_lexicon)
        }

        # Decision logic
        if pali_confidence >= threshold and pali_confidence > sinhala_confidence:
            return 'pali', {
                **debug_info,
                'classification': 'pali',
                'confidence': pali_confidence,
                'reason': f'Pali confidence {pali_confidence:.1%} > threshold {threshold:.1%} (Sinhala: {sinhala_confidence:.1%})'
            }

        elif sinhala_confidence >= threshold and sinhala_confidence > pali_confidence:
            return 'sinhala', {
                **debug_info,
                'classification': 'sinhala',
                'confidence': sinhala_confidence,
                'reason': f'Sinhala confidence {sinhala_confidence:.1%} > threshold {threshold:.1%} (Pali: {pali_confidence:.1%})'
            }

        else:
            return 'mixed', {
                **debug_info,
                'classification': 'mixed',
                'confidence': max(pali_confidence, sinhala_confidence),
                'reason': f'Below threshold or ambiguous (Pali: {pali_confidence:.1%}, Sinhala: {sinhala_confidence:.1%}, threshold: {threshold:.1%})'
            }

## 7. Load Extracted Pages

In [33]:
def load_extracted_corpus() -> Dict[str, Dict]:
    """
    Load all extracted pages from filtered_text directory.

    Returns:
        Dict with structure:
        {
            'book_id': {
                'metadata': {...},
                'pages': [
                    {
                        'page_num': 1,
                        'text': '...',
                        'file_path': '...'
                    }
                ]
            }
        }
    """

    print("\n" + "="*80)
    print("LOADING EXTRACTED CORPUS")
    print("="*80)

    # Load metadata
    metadata = load_json(METADATA_FILE, [])
    print(f"Loaded metadata for {len(metadata)} books")

    corpus = {}
    total_pages = 0

    # Only process books that were extracted and included in corpus
    included_books = [
        book for book in metadata
        if book.get('copyright_analysis', {}).get('can_include_in_corpus', False)
        and 'extraction' in book
    ]

    print(f"Found {len(included_books)} books to process\n")

    for book in tqdm(included_books, desc="Loading books"):
        book_name = book['book_name_si']
        category = book['Category']
        safe_name = sanitize_filename(book_name)

        # Get filtered text directory for this book
        book_dir = FILTERED_TEXT_DIR / category / safe_name

        if not book_dir.exists():
            continue

        # Load all pages
        page_files = sorted(book_dir.glob('page_*.txt'))
        pages = []

        for page_file in page_files:
            # Extract page number from filename
            page_num = int(page_file.stem.split('_')[1])

            # Load text
            try:
                with open(page_file, 'r', encoding='utf-8') as f:
                    text = f.read()

                pages.append({
                    'page_num': page_num,
                    'text': text,
                    'file_path': str(page_file.relative_to(BASE_DIR))
                })
                total_pages += 1
            except Exception as e:
                print(f"  Error loading {page_file}: {e}")

        if pages:
            corpus[book_name] = {
                'metadata': book,
                'pages': pages
            }

    print(f"\n✓ Loaded {len(corpus)} books with {total_pages} pages")
    print("="*80 + "\n")

    return corpus


# Load corpus
raw_corpus = load_extracted_corpus()


LOADING EXTRACTED CORPUS
Loaded metadata for 83 books
Found 16 books to process



Loading books:   0%|          | 0/16 [00:00<?, ?it/s]


✓ Loaded 16 books with 7075 pages



## 8. Clean and Process Corpus

In [35]:
def clean_and_classify_corpus(corpus: Dict, classifier: DualLexiconSinhalaPaliClassifier) -> Tuple[Dict, Dict]:
    """
    Clean text and classify language for all paragraphs in each page.
    """

    print("\n" + "="*80)
    print("CLEANING AND CLASSIFYING CORPUS (PARAGRAPH-LEVEL)")
    print("="*80)

    stats = {
        'total_pages': 0,
        'total_paragraphs': 0,
        'cleaned_paragraphs': 0,
        'removed_paragraphs': 0,
        'removal_reasons': defaultdict(int),
        'sinhala_paragraphs': 0,
        'pali_paragraphs': 0,
        'mixed_paragraphs': 0,
        'classification_reasons': defaultdict(int),
        'pages_kept': 0,
        'pages_removed': 0
    }

    cleaned_corpus = {}

    for book_name, book_data in tqdm(corpus.items(), desc="Processing books"):
        metadata = book_data['metadata']

        cleaned_pages = []

        for page in book_data['pages']:
            stats['total_pages'] += 1

            # --- STEP 1: CLEANING ---
            # Get raw text and clean it
            raw_text = page.get('text', '')
            cleaned_text = clean_text(raw_text)

            # --- STEP 2: SPLIT INTO PARAGRAPHS ---
            paragraphs_text = split_into_paragraphs(cleaned_text)

            classified_paragraphs = []

            # --- STEP 3: CLASSIFY EACH PARAGRAPH ---
            for p_text in paragraphs_text:
                stats['total_paragraphs'] += 1

                # Check if paragraph should be removed
                should_remove, remove_reason = should_remove_paragraph(p_text)

                if should_remove:
                    stats['removed_paragraphs'] += 1
                    stats['removal_reasons'][remove_reason] += 1
                    continue

                # Classify the paragraph
                classification, details = classifier.classify(p_text, threshold=CLASSIFICATION_THRESHOLD)

                # Update stats
                stats['cleaned_paragraphs'] += 1
                stats[f'{classification}_paragraphs'] += 1
                stats['classification_reasons'][details.get('reason', 'unknown')] += 1

                classified_paragraphs.append({
                    'text': p_text,
                    'language': classification,
                    'confidence': details.get('confidence', 0.0),
                    'char_count': len(p_text),
                    'word_count': len(p_text.split())
                })

            # --- STEP 4: PAGE LEVEL DECISIONS ---
            # Only keep page if it has at least one valid paragraph
            if classified_paragraphs:
                stats['pages_kept'] += 1

                # Calculate page-level language (majority vote)
                page_language = calculate_page_language(classified_paragraphs)

                # Calculate average confidence for the page
                avg_confidence = sum(p['confidence'] for p in classified_paragraphs) / len(classified_paragraphs)

                cleaned_pages.append({
                    'page_num': page['page_num'],
                    'file_path': page['file_path'], # Keep original file path reference
                    'paragraphs': classified_paragraphs,
                    'page_language': page_language,
                    'language_confidence': avg_confidence,
                    'language_reason': f"Majority vote of {len(classified_paragraphs)} paragraphs",
                    'paragraph_count': len(classified_paragraphs),
                    'char_count': sum(p['char_count'] for p in classified_paragraphs),
                    'sinhala_ratio': calculate_sinhala_ratio(cleaned_text)
                })
            else:
                stats['pages_removed'] += 1

        if cleaned_pages:
            cleaned_corpus[book_name] = {
                'metadata': book_data['metadata'],
                'pages': cleaned_pages
            }

    # Print statistics
    print_corpus_statistics(stats)

    return cleaned_corpus, dict(stats)


# ============================================
# HELPER FUNCTIONS
# ============================================

def split_into_paragraphs(text: str, min_length: int = 50) -> List[str]:
    """Split text into paragraphs."""
    # Split by double newlines (common paragraph separator)
    paragraphs = re.split(r'\n\s*\n', text)

    cleaned_paragraphs = []
    for para in paragraphs:
        para = para.strip()
        if len(para) >= min_length:
            cleaned_paragraphs.append(para)

    # If no double newlines found, try single newlines
    if not cleaned_paragraphs:
        paragraphs = text.split('\n')
        cleaned_paragraphs = [p.strip() for p in paragraphs if len(p.strip()) >= min_length]

    return cleaned_paragraphs


def should_remove_paragraph(text: str, min_length: int = 30, max_length: int = 10000) -> Tuple[bool, str]:
    """Check if a paragraph should be removed."""
    if len(text) < min_length:
        return True, 'paragraph_too_short'
    if len(text) > max_length:
        return True, 'paragraph_too_long'

    # Mostly numbers or punctuation check can be added here
    return False, None


def calculate_page_language(paragraphs: List[Dict]) -> str:
    """Determine page-level language based on paragraph classifications."""
    if not paragraphs:
        return 'mixed'

    sinhala_score = sum(1 for p in paragraphs if p['language'] == 'sinhala')
    pali_score = sum(1 for p in paragraphs if p['language'] == 'pali')
    total = len(paragraphs)

    if sinhala_score / total > 0.6:
        return 'sinhala'
    elif pali_score / total > 0.6:
        return 'pali'
    else:
        return 'mixed'


def print_corpus_statistics(stats: Dict):
    """Print detailed corpus statistics."""
    print(f"\n{'='*80}")
    print("PAGE-LEVEL STATISTICS")
    print('='*80)
    print(f"Total pages processed: {stats['total_pages']}")
    print(f"Pages kept: {stats['pages_kept']}")
    print(f"Pages removed: {stats['pages_removed']}")

    print(f"\n{'='*80}")
    print("LANGUAGE CLASSIFICATION")
    print('='*80)
    print(f"Sinhala paragraphs: {stats['sinhala_paragraphs']}")
    print(f"Pali paragraphs: {stats['pali_paragraphs']}")
    print(f"Mixed paragraphs: {stats['mixed_paragraphs']}")
    print('='*80 + '\n')


# ============================================
# EXECUTION
# ============================================

# Initialize classifier
language_classifier = DualLexiconSinhalaPaliClassifier(
    pali_lexicon_path=str(PALI_LEXICON),
    auto_download_sinhala=True
)

# Clean and classify
cleaned_corpus, cleaning_stats = clean_and_classify_corpus(
    raw_corpus,
    language_classifier
)

✓ Lexicon downloaded to: sinhala_lexicon.tsv
✓ Loaded 41,617 Sinhala words
Loading Pali lexicon from pali_lexicons_single.tsv...
✓ Loaded 14,278 Pali words

CLASSIFIER INITIALIZED
Sinhala lexicon: 41,617 words
  Source: auto-downloaded: sinhala_lexicon.tsv
Pali lexicon:    14,278 words
  Source: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/pali_lexicon/pali_lexicons_single.tsv


CLEANING AND CLASSIFYING CORPUS (PARAGRAPH-LEVEL)


Processing books:   0%|          | 0/16 [00:00<?, ?it/s]


PAGE-LEVEL STATISTICS
Total pages processed: 7075
Pages kept: 7064
Pages removed: 11

LANGUAGE CLASSIFICATION
Sinhala paragraphs: 1385
Pali paragraphs: 0
Mixed paragraphs: 5679



## 9. Save Corpus - Multiple Formats

In [37]:
def save_corpus_multi_format(corpus: Dict) -> Dict:
    """
    Save corpus in multiple formats:
    1. Individual page files (by language)
    2. Book-level files
    3. Consolidated corpus files
    4. CSV metadata

    Returns:
        Statistics about saved files
    """

    print("\n" + "="*80)
    print("SAVING CORPUS - MULTIPLE FORMATS")
    print("="*80)

    stats = {
        'individual_files': 0,
        'book_files': 0,
        'consolidated_files': 0
    }

    # Data structures for different outputs
    page_records = []  # For CSV
    book_records = []  # For CSV
    language_texts = {  # For consolidated files
        'sinhala': [],
        'pali': [],
        'mixed': [],
        'full': []
    }

    print("\n[1/4] Saving individual page files...")
    # print(pages_df.head())

    for book_name, book_data in tqdm(corpus.items(), desc="Processing books"):
        metadata = book_data['metadata']
        category = metadata['Category']
        safe_name = sanitize_filename(book_name)

        # Book-level statistics
        book_stats = {
            'sinhala': 0,
            'pali': 0,
            'mixed': 0,
            'total_chars': 0
        }
        book_text = []

        for page in book_data['pages']:
            # print(page)
            language = page['page_language']
            text = ''
            for para in page['paragraphs']:
                text = text+para['text']
            # text = page['page_text']

            # Update book stats
            book_stats[language] += 1
            book_stats['total_chars'] += len(text)
            book_text.append(text)

            # Save individual page file (by language)
            page_filename = f"page_{page['page_num']:03d}.txt"
            page_dir = BY_LANGUAGE_DIR / language / category / safe_name
            page_dir.mkdir(parents=True, exist_ok=True)
            page_path = page_dir / page_filename

            with open(page_path, 'w', encoding='utf-8') as f:
                f.write(text)
            stats['individual_files'] += 1

            # Add to consolidated corpus
            language_texts[language].append(text)
            language_texts['full'].append(text)

            # Add to page records (for CSV)
            page_records.append({
                'page_id': f"{safe_name}_p{page['page_num']:03d}",
                'book_name_si': book_name,
                'book_name_en': metadata.get('book_name_en', ''),
                'category': category,
                'page_num': page['page_num'],
                'language': language,
                'language_confidence': page['language_confidence'],
                'language_reason': page['language_reason'],
                'char_count': page['char_count'],
                'sinhala_ratio': page['sinhala_ratio'],
                'file_path': str(page_path.relative_to(CORPUS_DIR))
            })

        # Save book-level concatenation
        book_file_dir = BY_BOOK_DIR / category
        book_file_dir.mkdir(parents=True, exist_ok=True)
        book_file_path = book_file_dir / f"{safe_name}.txt"

        with open(book_file_path, 'w', encoding='utf-8') as f:
            f.write('\n\n[PAGE_BREAK]\n\n'.join(book_text))
        stats['book_files'] += 1

        # Add to book records (for CSV)
        book_records.append({
            'book_name_si': book_name,
            'book_name_en': metadata.get('book_name_en', ''),
            'category': category,
            'author': metadata.get('Author', {}).get('name', ''),
            'total_pages': len(book_data['pages']),
            'sinhala_pages': book_stats['sinhala'],
            'pali_pages': book_stats['pali'],
            'mixed_pages': book_stats['mixed'],
            'total_chars': book_stats['total_chars'],
            'file_path': str(book_file_path.relative_to(CORPUS_DIR))
        })

    print(f"  ✓ Saved {stats['individual_files']} individual page files")
    print(f"  ✓ Saved {stats['book_files']} book-level files")

    # Save consolidated corpus files
    print("\n[2/4] Saving consolidated corpus files...")

    for lang, texts in language_texts.items():
        if not texts:
            continue

        corpus_file = CONSOLIDATED_DIR / f"{lang}_corpus.txt"
        with open(corpus_file, 'w', encoding='utf-8') as f:
            f.write('\n\n'.join(texts))
        stats['consolidated_files'] += 1

        file_size = corpus_file.stat().st_size / (1024 * 1024)  # MB
        print(f"  ✓ {lang}_corpus.txt: {len(texts)} pages, {file_size:.2f} MB")

    # Save CSV files
    print("\n[3/4] Saving CSV metadata...")

    pages_df = pd.DataFrame(page_records)
    pages_csv = STRUCTURED_DIR / "corpus_pages.csv"
    pages_df.to_csv(pages_csv, index=False, encoding='utf-8-sig')
    print(f"  ✓ corpus_pages.csv: {len(pages_df)} rows")

    books_df = pd.DataFrame(book_records)
    books_csv = STRUCTURED_DIR / "corpus_books.csv"
    books_df.to_csv(books_csv, index=False, encoding='utf-8-sig')
    print(f"  ✓ corpus_books.csv: {len(books_df)} rows")

    # Generate statistics
    print("\n[4/4] Generating corpus statistics...")

    corpus_stats = {
        'total_books': len(corpus),
        'total_pages': len(page_records),
        'language_distribution': {
            'sinhala': {
                'pages': pages_df[pages_df['language'] == 'sinhala'].shape[0],
                'chars': int(pages_df[pages_df['language'] == 'sinhala']['char_count'].sum())
            },
            'pali': {
                'pages': pages_df[pages_df['language'] == 'pali'].shape[0],
                'chars': int(pages_df[pages_df['language'] == 'pali']['char_count'].sum())
            },
            'mixed': {
                'pages': pages_df[pages_df['language'] == 'mixed'].shape[0],
                'chars': int(pages_df[pages_df['language'] == 'mixed']['char_count'].sum())
            }
        },
        'category_distribution': books_df.groupby('category').size().to_dict(),
        'avg_pages_per_book': pages_df.groupby('book_name_si').size().mean(),
        'avg_chars_per_page': pages_df['char_count'].mean()
    }

    stats_file = CORPUS_METADATA_DIR / "corpus_statistics.json"
    save_json(corpus_stats, stats_file)
    print(f"  ✓ corpus_statistics.json saved")

    print("\n" + "="*80)
    print("✓ ALL FORMATS SAVED")
    print("="*80 + "\n")

    return stats, pages_df, books_df, corpus_stats


# Save corpus in all formats
save_stats, pages_df, books_df, corpus_stats = save_corpus_multi_format(cleaned_corpus)


SAVING CORPUS - MULTIPLE FORMATS

[1/4] Saving individual page files...


Processing books:   0%|          | 0/16 [00:00<?, ?it/s]

  ✓ Saved 7064 individual page files
  ✓ Saved 16 book-level files

[2/4] Saving consolidated corpus files...
  ✓ sinhala_corpus.txt: 1385 pages, 5.84 MB
  ✓ mixed_corpus.txt: 5679 pages, 23.76 MB
  ✓ full_corpus.txt: 7064 pages, 29.61 MB

[3/4] Saving CSV metadata...
  ✓ corpus_pages.csv: 7064 rows
  ✓ corpus_books.csv: 16 rows

[4/4] Generating corpus statistics...
  ✓ corpus_statistics.json saved

✓ ALL FORMATS SAVED



In [38]:
def count_words(text: str) -> int:
    """
    Count words in Sinhala/Pali text.
    Uses whitespace tokenization.
    """
    return len(text.split())

def generate_detailed_statistics(pages_df: pd.DataFrame, corpus_dir: Path) -> Dict:
    """
    Generate comprehensive statistics including word counts.
    """

    print("\n" + "="*80)
    print("GENERATING DETAILED CORPUS STATISTICS")
    print("="*80)

    detailed_stats = {
        'by_language': {},
        'by_category': {},
        'by_book': {},
        'overall': {}
    }

    print("\n[1/4] Calculating language-level statistics...")

    for language in ['sinhala', 'pali', 'mixed']:
        lang_pages = pages_df[pages_df['language'] == language]

        if len(lang_pages) == 0:
            continue

        print(f"\n  Processing {language}...")

        # Load consolidated corpus file
        corpus_file = CONSOLIDATED_DIR / f"{language}_corpus.txt"

        if corpus_file.exists():
            with open(corpus_file, 'r', encoding='utf-8') as f:
                full_text = f.read()

            word_count = count_words(full_text)
            char_count = len(full_text)

            detailed_stats['by_language'][language] = {
                'total_pages': len(lang_pages),
                'total_books': lang_pages['book_name_si'].nunique(),
                'total_words': word_count,
                'total_characters': char_count,
                'avg_words_per_page': word_count / len(lang_pages) if len(lang_pages) > 0 else 0,
                'avg_chars_per_page': char_count / len(lang_pages) if len(lang_pages) > 0 else 0,
                'avg_words_per_book': word_count / lang_pages['book_name_si'].nunique() if lang_pages['book_name_si'].nunique() > 0 else 0,
            }

            print(f"    Pages: {len(lang_pages):,}")
            print(f"    Books: {lang_pages['book_name_si'].nunique()}")
            print(f"    Words: {word_count:,}")
            print(f"    Characters: {char_count:,}")

    print("\n[2/4] Calculating category-level statistics...")

    for category in pages_df['category'].unique():
        cat_pages = pages_df[pages_df['category'] == category]

        category_stats = {
            'total_pages': len(cat_pages),
            'total_books': cat_pages['book_name_si'].nunique(),
            'languages': {}
        }

        # Count by language within category
        for language in ['sinhala', 'pali', 'mixed']:
            lang_cat_pages = cat_pages[cat_pages['language'] == language]

            if len(lang_cat_pages) > 0:
                # Calculate total words from all pages in this category+language
                total_words = 0
                total_chars = 0

                for _, row in lang_cat_pages.iterrows():
                    file_path = CORPUS_DIR / row['file_path']
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text = f.read()
                        total_words += count_words(text)
                        total_chars += len(text)
                    except:
                        pass

                category_stats['languages'][language] = {
                    'pages': len(lang_cat_pages),
                    'words': total_words,
                    'characters': total_chars
                }

        detailed_stats['by_category'][category] = category_stats

        print(f"  {category}: {len(cat_pages)} pages, {cat_pages['book_name_si'].nunique()} books")


    print("\n[3/4] Calculating book-level statistics...")

    book_stats_list = []

    for book_name in tqdm(pages_df['book_name_si'].unique(), desc="  Processing books"):
        book_pages = pages_df[pages_df['book_name_si'] == book_name]

        book_stat = {
            'book_name_si': book_name,
            'book_name_en': book_pages.iloc[0]['book_name_en'],
            'category': book_pages.iloc[0]['category'],
            'total_pages': len(book_pages),
            'languages': {}
        }

        # Count by language
        for language in ['sinhala', 'pali', 'mixed']:
            lang_pages = book_pages[book_pages['language'] == language]

            if len(lang_pages) > 0:
                total_words = 0
                total_chars = 0

                for _, row in lang_pages.iterrows():
                    file_path = CORPUS_DIR / row['file_path']
                    try:
                        with open(file_path, 'r', encoding='utf-8') as f:
                            text = f.read()
                        total_words += count_words(text)
                        total_chars += len(text)
                    except:
                        pass

                book_stat['languages'][language] = {
                    'pages': len(lang_pages),
                    'words': total_words,
                    'characters': total_chars
                }

        book_stats_list.append(book_stat)

    detailed_stats['by_book'] = book_stats_list


    print("\n[4/4] Calculating overall statistics...")

    total_words = sum(
        detailed_stats['by_language'].get(lang, {}).get('total_words', 0)
        for lang in ['sinhala', 'pali', 'mixed']
    )

    total_chars = sum(
        detailed_stats['by_language'].get(lang, {}).get('total_characters', 0)
        for lang in ['sinhala', 'pali', 'mixed']
    )

    detailed_stats['overall'] = {
        'total_books': pages_df['book_name_si'].nunique(),
        'total_pages': len(pages_df),
        'total_words': total_words,
        'total_characters': total_chars,
        'avg_words_per_page': total_words / len(pages_df) if len(pages_df) > 0 else 0,
        'avg_chars_per_page': total_chars / len(pages_df) if len(pages_df) > 0 else 0,
    }

    print("\n" + "="*80)
    print("DETAILED STATISTICS COMPLETE")
    print("="*80)

    return detailed_stats


# Generate detailed statistics
detailed_stats = generate_detailed_statistics(pages_df, CORPUS_DIR)

# Save detailed statistics
detailed_stats_file = CORPUS_METADATA_DIR / "detailed_corpus_statistics.json"
save_json(detailed_stats, detailed_stats_file)
print(f"\n✓ Saved to: {detailed_stats_file}")

print("\n" + "="*80)
print("COMPREHENSIVE CORPUS SUMMARY")
print("="*80)

print("\n📊 OVERALL CORPUS:")
print("-" * 80)
overall = detailed_stats['overall']
print(f"  Total Books:      {overall['total_books']:,}")
print(f"  Total Pages:      {overall['total_pages']:,}")
print(f"  Total Words:      {overall['total_words']:,}")
print(f"  Total Characters: {overall['total_characters']:,}")
print(f"  Avg Words/Page:   {overall['avg_words_per_page']:.1f}")

print("\n📚 BY LANGUAGE:")
print("-" * 80)

for language in ['sinhala', 'pali', 'mixed']:
    if language in detailed_stats['by_language']:
        stats = detailed_stats['by_language'][language]

        # Calculate percentages
        word_pct = (stats['total_words'] / overall['total_words'] * 100) if overall['total_words'] > 0 else 0
        page_pct = (stats['total_pages'] / overall['total_pages'] * 100) if overall['total_pages'] > 0 else 0

        print(f"\n  {language.upper()}:")
        print(f"    Books:      {stats['total_books']:,}")
        print(f"    Pages:      {stats['total_pages']:,} ({page_pct:.1f}%)")
        print(f"    Words:      {stats['total_words']:,} ({word_pct:.1f}%)")
        print(f"    Characters: {stats['total_characters']:,}")
        print(f"    Avg Words/Page: {stats['avg_words_per_page']:.1f}")

print("\n📁 BY CATEGORY:")
print("-" * 80)

for category, stats in sorted(detailed_stats['by_category'].items()):
    print(f"\n  {category}:")
    print(f"    Books: {stats['total_books']:,}, Pages: {stats['total_pages']:,}")

    for language, lang_stats in stats['languages'].items():
        print(f"    {language.capitalize()}: {lang_stats['pages']} pages, {lang_stats['words']:,} words")

print("\n📖 TOP 10 BOOKS BY WORD COUNT:")
print("-" * 80)

# Sort books by total words
books_with_words = []
for book in detailed_stats['by_book']:
    total_words = sum(
        lang_stats['words']
        for lang_stats in book['languages'].values()
    )
    books_with_words.append((book['book_name_si'], total_words, book['total_pages']))

books_with_words.sort(key=lambda x: x[1], reverse=True)

for i, (book_name, word_count, page_count) in enumerate(books_with_words[:10], 1):
    print(f"  {i:2d}. {book_name[:60]:60s} {word_count:>8,} words ({page_count} pages)")

print("\n" + "="*80)


print("\nCreating summary tables...")

# Language summary table
language_summary = []
for language in ['sinhala', 'pali', 'mixed']:
    if language in detailed_stats['by_language']:
        stats = detailed_stats['by_language'][language]
        language_summary.append({
            'language': language,
            'books': stats['total_books'],
            'pages': stats['total_pages'],
            'words': stats['total_words'],
            'characters': stats['total_characters'],
            'avg_words_per_page': round(stats['avg_words_per_page'], 1),
            'avg_chars_per_page': round(stats['avg_chars_per_page'], 1)
        })

language_summary_df = pd.DataFrame(language_summary)
language_summary_csv = STRUCTURED_DIR / "corpus_language_summary.csv"
language_summary_df.to_csv(language_summary_csv, index=False)
print(f"  ✓ {language_summary_csv.name}")

# Book statistics table with word counts
book_summary = []
for book in detailed_stats['by_book']:
    total_words = sum(lang_stats['words'] for lang_stats in book['languages'].values())
    total_chars = sum(lang_stats['characters'] for lang_stats in book['languages'].values())

    book_summary.append({
        'book_name_si': book['book_name_si'],
        'book_name_en': book['book_name_en'],
        'category': book['category'],
        'total_pages': book['total_pages'],
        'total_words': total_words,
        'total_characters': total_chars,
        'sinhala_pages': book['languages'].get('sinhala', {}).get('pages', 0),
        'sinhala_words': book['languages'].get('sinhala', {}).get('words', 0),
        'pali_pages': book['languages'].get('pali', {}).get('pages', 0),
        'pali_words': book['languages'].get('pali', {}).get('words', 0),
        'mixed_pages': book['languages'].get('mixed', {}).get('pages', 0),
        'mixed_words': book['languages'].get('mixed', {}).get('words', 0),
    })

book_summary_df = pd.DataFrame(book_summary)
book_summary_df = book_summary_df.sort_values('total_words', ascending=False)
book_summary_csv = STRUCTURED_DIR / "corpus_book_statistics.csv"
book_summary_df.to_csv(book_summary_csv, index=False, encoding='utf-8-sig')
print(f"  ✓ {book_summary_csv.name}")

print("\n✓ All statistics generated and saved!")
print("="*80 + "\n")


GENERATING DETAILED CORPUS STATISTICS

[1/4] Calculating language-level statistics...

  Processing sinhala...
    Pages: 1,385
    Books: 12
    Words: 415,439
    Characters: 2,348,214

  Processing mixed...
    Pages: 5,679
    Books: 16
    Words: 1,588,535
    Characters: 9,636,104

[2/4] Calculating category-level statistics...
  books-related-to-the-tipitaka: 2631 pages, 5 books
  old-books: 4226 pages, 8 books
  buddhist-characters: 207 pages, 3 books

[3/4] Calculating book-level statistics...


  Processing books:   0%|          | 0/16 [00:00<?, ?it/s]


[4/4] Calculating overall statistics...

DETAILED STATISTICS COMPLETE

✓ Saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus/metadata/detailed_corpus_statistics.json

COMPREHENSIVE CORPUS SUMMARY

📊 OVERALL CORPUS:
--------------------------------------------------------------------------------
  Total Books:      16
  Total Pages:      7,064
  Total Words:      2,003,974
  Total Characters: 11,984,318
  Avg Words/Page:   283.7

📚 BY LANGUAGE:
--------------------------------------------------------------------------------

  SINHALA:
    Books:      12
    Pages:      1,385 (19.6%)
    Words:      415,439 (20.7%)
    Characters: 2,348,214
    Avg Words/Page: 300.0

  MIXED:
    Books:      16
    Pages:      5,679 (80.4%)
    Words:      1,588,535 (79.3%)
    Characters: 9,636,104
    Avg Words/Page: 279.7

📁 BY CATEGORY:
--------------------------------------------------------------------------------

  books-related-to-the-tipitaka

## 10. Create Train/Validation/Test Splits

In [39]:
def create_train_val_test_splits(books_df: pd.DataFrame,
                                  pages_df: pd.DataFrame,
                                  split_ratios: Dict[str, float]) -> Dict:
    """
    Create train/validation/test splits at BOOK level.
    Ensures no data leakage (same book never in multiple splits).
    """

    print("\n" + "="*80)
    print("CREATING TRAIN/VALIDATION/TEST SPLITS")
    print("="*80)
    print(f"Split ratios: {split_ratios}\n")

    import numpy as np

    splits = {}

    # Process each language separately
    for language in ['sinhala', 'pali', 'mixed']:
        print(f"\n[{language.upper()}]")

        # Get pages for this language
        lang_pages = pages_df[pages_df['language'] == language]

        if len(lang_pages) == 0:
            print(f"  No pages found, skipping")
            continue

        # Get unique books
        unique_books = lang_pages['book_name_si'].unique()
        np.random.seed(42)  # For reproducibility
        np.random.shuffle(unique_books)

        # Calculate split indices
        n_books = len(unique_books)
        train_end = int(n_books * split_ratios['train'])
        val_end = train_end + int(n_books * split_ratios['validation'])

        # Split books
        train_books = unique_books[:train_end]
        val_books = unique_books[train_end:val_end]
        test_books = unique_books[val_end:]

        # Get pages for each split
        train_pages = lang_pages[lang_pages['book_name_si'].isin(train_books)]
        val_pages = lang_pages[lang_pages['book_name_si'].isin(val_books)]
        test_pages = lang_pages[lang_pages['book_name_si'].isin(test_books)]

        print(f"  Books: {len(train_books)} train, {len(val_books)} val, {len(test_books)} test")
        print(f"  Pages: {len(train_pages)} train, {len(val_pages)} val, {len(test_pages)} test")

        # Save split files
        lang_dir = SPLITS_DIR / language

        for split_name, split_pages in [('train', train_pages),
                                         ('validation', val_pages),
                                         ('test', test_pages)]:
            if len(split_pages) == 0:
                continue

            # Load actual text for each page
            texts = []
            for _, row in split_pages.iterrows():
                file_path = CORPUS_DIR / row['file_path']
                with open(file_path, 'r', encoding='utf-8') as f:
                    texts.append(f.read())

            # Save split file
            split_file = lang_dir / f"{split_name}.txt"
            with open(split_file, 'w', encoding='utf-8') as f:
                f.write('\n\n'.join(texts))

            file_size = split_file.stat().st_size / (1024 * 1024)  # MB
            print(f"  ✓ {split_name}.txt: {len(texts)} pages, {file_size:.2f} MB")

        splits[language] = {
            'train_books': train_books.tolist(),
            'val_books': val_books.tolist(),
            'test_books': test_books.tolist(),
            'train_pages': len(train_pages),
            'val_pages': len(val_pages),
            'test_pages': len(test_pages)
        }

    # Save split information
    splits_info_file = CORPUS_METADATA_DIR / "train_val_test_splits.json"
    save_json(splits, splits_info_file)

    print("\n" + "="*80)
    print("✓ SPLITS CREATED")
    print("="*80 + "\n")

    return splits


# Create splits
splits_info = create_train_val_test_splits(books_df, pages_df, SPLIT_RATIOS)


CREATING TRAIN/VALIDATION/TEST SPLITS
Split ratios: {'train': 0.8, 'validation': 0.1, 'test': 0.1}


[SINHALA]
  Books: 9 train, 1 val, 2 test
  Pages: 1328 train, 1 val, 56 test
  ✓ train.txt: 1328 pages, 5.63 MB
  ✓ validation.txt: 1 pages, 0.00 MB
  ✓ test.txt: 56 pages, 0.21 MB

[PALI]
  No pages found, skipping

[MIXED]
  Books: 12 train, 1 val, 3 test
  Pages: 3653 train, 770 val, 1256 test
  ✓ train.txt: 3653 pages, 15.40 MB
  ✓ validation.txt: 770 pages, 3.28 MB
  ✓ test.txt: 1256 pages, 5.08 MB

✓ SPLITS CREATED



## 11. Generate Corpus Manifest

In [41]:
def generate_corpus_manifest(corpus_stats: Dict,
                              cleaning_stats: Dict,
                              splits_info: Dict) -> Dict:
    """
    Generate comprehensive corpus manifest documenting the entire corpus.
    """

    manifest = {
        'corpus_info': {
            'name': 'Sinhala Buddhist Texts Corpus',
            'version': '1.0',
            'created_at': datetime.now().isoformat(),
            'description': 'Cleaned and classified corpus of public domain Sinhala Buddhist texts',
            'language': ['sin', 'pli'],  # ISO 639-3 codes
            'script': 'Sinh',  # ISO 15924 code
        },

        'statistics': corpus_stats,

        'cleaning_process': {
            'total_pages_input': cleaning_stats.get('total_pages', 0),
            # FIX: Changed 'cleaned_pages' to 'pages_kept'
            'pages_kept': cleaning_stats.get('pages_kept', 0),
            # FIX: Changed 'removed_pages' to 'pages_removed'
            'pages_removed': cleaning_stats.get('pages_removed', 0),
            'removal_reasons': dict(cleaning_stats.get('removal_reasons', {})),
            'parameters': {
                'english_threshold': ENGLISH_THRESHOLD,
                'min_page_length': MIN_PAGE_LENGTH,
                'classification_threshold': CLASSIFICATION_THRESHOLD
            },
            'cleaning_rules': [
                'Removed URLs',
                'Removed email addresses',
                'Removed isolated page numbers',
                'Removed control characters',
                'Normalized Unicode to NFC',
                'Normalized whitespace',
                'Removed duplicate lines',
                'Filtered excessive English content'
            ]
        },

        'language_classification': {
            'classifier': 'SinhalaPaliClassifier',
            'method': 'Grammatical pattern matching',
            'threshold': CLASSIFICATION_THRESHOLD,
            'categories': ['sinhala', 'pali', 'mixed'],
            'distribution': dict(cleaning_stats.get('classification_reasons', {}))
        },

        'train_val_test_splits': splits_info,

        'file_structure': {
            'by_language': 'Individual pages organized by language',
            'by_book': 'Book-level concatenations',
            'consolidated': 'Language-specific corpus files for training',
            'splits': 'Train/validation/test splits by language',
            'structured': 'CSV files for analysis (pages and books)',
            'metadata': 'Corpus documentation'
        },

        'copyright': {
            'status': 'Public Domain',
            'criteria': '70-year rule from author death (Sri Lankan IP Act No. 36 of 2003)',
            'cutoff_year': 1954,
            'verification_date': datetime.now().isoformat(),
            'note': 'All included texts are from authors who died before 1955'
        },

        'citation': {
            'source': 'International Foundation for Buddhist Communication (IFBC)',
            'url': 'https://download.ifbcnet.org/',
            'suggested_citation': 'Sinhala Buddhist Texts Corpus v1.0, compiled from IFBC digital archive (2025)'
        },

        'usage_notes': {
            'recommended_for': [
                'Sinhala language modeling',
                'Buddhist text analysis',
                'Historical Sinhala NLP',
                'Pali transliteration studies'
            ],
            'limitations': [
                'OCR quality varies by document',
                'Some pages may contain mixed languages',
                'Classical/archaic Sinhala usage'
            ]
        }
    }

    return manifest


# Generate manifest
# Ensure cleaning_stats uses defaults if keys are missing to prevent crashes
manifest = generate_corpus_manifest(corpus_stats, cleaning_stats, splits_info)

# Save manifest
manifest_file = CORPUS_METADATA_DIR / "corpus_manifest.json"
save_json(manifest, manifest_file)

print("\n" + "="*80)
print("CORPUS MANIFEST GENERATED")
print("="*80)
print(f"✓ Saved to: {manifest_file}")
print("\nKey Statistics:")
print(f"  Total books: {corpus_stats.get('total_books', 0)}")
print(f"  Total pages: {corpus_stats.get('total_pages', 0)}")
print(f"  Sinhala pages: {corpus_stats.get('language_distribution', {}).get('sinhala', {}).get('pages', 0)}")
print(f"  Pali pages: {corpus_stats.get('language_distribution', {}).get('pali', {}).get('pages', 0)}")
print(f"  Mixed pages: {corpus_stats.get('language_distribution', {}).get('mixed', {}).get('pages', 0)}")
print("="*80 + "\n")


CORPUS MANIFEST GENERATED
✓ Saved to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus/metadata/corpus_manifest.json

Key Statistics:
  Total books: 16
  Total pages: 7064
  Sinhala pages: 1385
  Pali pages: 0
  Mixed pages: 5679



## 12. Final Summary and Validation

In [42]:
print("\n" + "="*80)
print("CORPUS CREATION COMPLETE")
print("="*80)

print("\n📁 Output Directory Structure:")
print(f"   {CORPUS_DIR}/")
print("   ├── raw/")
print("   │   ├── by_language/{sinhala,pali,mixed}/category/book/page_XXX.txt")
print("   │   └── by_book/category/book.txt")
print("   ├── consolidated/")
print("   │   ├── sinhala_corpus.txt")
print("   │   ├── pali_corpus.txt")
print("   │   ├── mixed_corpus.txt")
print("   │   └── full_corpus.txt")
print("   ├── splits/")
print("   │   └── {sinhala,pali,mixed}/{train,validation,test}.txt")
print("   ├── structured/")
print("   │   ├── corpus_pages.csv")
print("   │   └── corpus_books.csv")
print("   └── metadata/")
print("       ├── corpus_manifest.json")
print("       ├── corpus_statistics.json")
print("       └── train_val_test_splits.json")

print("\n📊 Files Created:")
print(f"   Individual page files: {save_stats['individual_files']}")
print(f"   Book-level files: {save_stats['book_files']}")
print(f"   Consolidated corpus files: {save_stats['consolidated_files']}")
print(f"   CSV files: 2 (pages and books)")
print(f"   Metadata files: 3 (manifest, statistics, splits)")

print("\n🎯 Ready for Next Steps:")
print("   ✓ Masked word prediction (use splits/sinhala/train.txt)")
print("   ✓ Perplexity evaluation (use splits/sinhala/test.txt)")
print("   ✓ Language model training (use consolidated/sinhala_corpus.txt)")
print("   ✓ Corpus analysis (use structured/*.csv files)")

print("\n" + "="*80)
print("🎉 SUCCESS!")
print("="*80 + "\n")


CORPUS CREATION COMPLETE

📁 Output Directory Structure:
   /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/corpus/
   ├── raw/
   │   ├── by_language/{sinhala,pali,mixed}/category/book/page_XXX.txt
   │   └── by_book/category/book.txt
   ├── consolidated/
   │   ├── sinhala_corpus.txt
   │   ├── pali_corpus.txt
   │   ├── mixed_corpus.txt
   │   └── full_corpus.txt
   ├── splits/
   │   └── {sinhala,pali,mixed}/{train,validation,test}.txt
   ├── structured/
   │   ├── corpus_pages.csv
   │   └── corpus_books.csv
   └── metadata/
       ├── corpus_manifest.json
       ├── corpus_statistics.json
       └── train_val_test_splits.json

📊 Files Created:
   Individual page files: 7064
   Book-level files: 16
   Consolidated corpus files: 3
   CSV files: 2 (pages and books)
   Metadata files: 3 (manifest, statistics, splits)

🎯 Ready for Next Steps:
   ✓ Masked word prediction (use splits/sinhala/train.txt)
   ✓ Perplexity evaluation (use splits/sinhala/

## 13. Optional: Export Sample for Inspection

In [ ]:
# Export samples for manual inspection
print("Creating sample exports for manual inspection...\n")

for language in ['sinhala', 'pali', 'mixed']:
    lang_pages = pages_df[pages_df['language'] == language]

    if len(lang_pages) == 0:
        continue

    # Sample 10 pages
    samples = lang_pages.sample(min(10, len(lang_pages)), random_state=42)

    print(f"\n{language.upper()} Samples:")
    print("="*80)

    for idx, (_, row) in enumerate(samples.iterrows(), 1):
        file_path = CORPUS_DIR / row['file_path']
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()

        print(f"\nSample {idx}:")
        print(f"  Book: {row['book_name_si']}")
        print(f"  Page: {row['page_num']}")
        print(f"  Confidence: {row['language_confidence']:.2f}")
        print(f"  Reason: {row['language_reason']}")
        print(f"  Preview: {text[:200]}...")
        print("-"*80)

print("\n✓ Sample inspection complete")

# Checking for sinhala word overlaps in pali lexicon

In [ ]:
# """
# Lexicon Overlap Analysis and Filtering
# ======================================

# This script:
# 1. Loads Sinhala lexicon (Google)
# 2. Loads Pali lexicon (your custom one)
# 3. Finds overlapping words
# 4. Creates filtered Pali lexicon (no Sinhala words)
# 5. Generates detailed reports

# Author: Multi-Lingual Buddhist Conversational Agent Project
# """

# import pandas as pd
# from pathlib import Path
# from typing import Set, Dict, Tuple
# from collections import Counter


# def load_sinhala_lexicon(tsv_path: str) -> Set[str]:
#     """
#     Load Sinhala words from Google's TSV lexicon.

#     Format: word<TAB>phonemic_transcription
#     """
#     lexicon = set()

#     with open(tsv_path, 'r', encoding='utf-8') as f:
#         for line in f:
#             line = line.strip()

#             # Skip comments and empty lines
#             if not line or line.startswith('#'):
#                 continue

#             # Extract first column (word)
#             parts = line.split('\t')
#             if len(parts) >= 1:
#                 word = parts[0].strip()
#                 if word and word != 'word':  # Skip header
#                     lexicon.add(word)

#     return lexicon


# def load_pali_lexicon(tsv_path: str) -> Set[str]:
#     """
#     Load Pali words from your custom lexicon.

#     Supports TSV with 'word' column or plain text.
#     """
#     path = Path(tsv_path)

#     try:
#         # Try loading as TSV
#         df = pd.read_csv(path, sep='\t', encoding='utf-8')

#         if 'word' in df.columns:
#             lexicon = set(df['word'].dropna().astype(str).tolist())
#         else:
#             # Use first column
#             lexicon = set(df.iloc[:, 0].dropna().astype(str).tolist())

#     except Exception:
#         # Fallback: plain text (one word per line)
#         lexicon = set()
#         with open(path, 'r', encoding='utf-8') as f:
#             for line in f:
#                 word = line.strip()
#                 if word and not word.startswith('#'):
#                     lexicon.add(word)

#     return lexicon


# def analyze_overlap(sinhala_lex: Set[str], pali_lex: Set[str]) -> Dict:
#     """
#     Analyze overlap between Sinhala and Pali lexicons.

#     Returns:
#         Dictionary with overlap analysis
#     """
#     # Find overlapping words
#     overlap = sinhala_lex & pali_lex

#     # Find unique words
#     sinhala_only = sinhala_lex - pali_lex
#     pali_only = pali_lex - sinhala_lex

#     # Calculate statistics
#     total_unique = len(sinhala_only) + len(pali_only) + len(overlap)
#     overlap_percentage = (len(overlap) / len(pali_lex) * 100) if pali_lex else 0

#     analysis = {
#         'sinhala_total': len(sinhala_lex),
#         'pali_total': len(pali_lex),
#         'overlap_count': len(overlap),
#         'overlap_percentage': overlap_percentage,
#         'sinhala_only': len(sinhala_only),
#         'pali_only': len(pali_only),
#         'total_unique_words': total_unique,
#         'overlapping_words': sorted(overlap),
#         'pali_unique_words': sorted(pali_only)
#     }

#     return analysis


# def save_filtered_pali_lexicon(pali_only: Set[str], output_path: str, original_tsv: str = None):
#     """
#     Save filtered Pali lexicon (no Sinhala overlap) in multiple formats.

#     Args:
#         pali_only: Set of Pali-only words
#         output_path: Base path for output files
#         original_tsv: Optional path to original Pali TSV to preserve metadata
#     """
#     output_base = Path(output_path).stem
#     output_dir = Path(output_path).parent

#     # 1. Save as simple word list (.txt)
#     txt_path = output_dir / f"{output_base}_filtered.txt"
#     with open(txt_path, 'w', encoding='utf-8') as f:
#         for word in sorted(pali_only):
#             f.write(f"{word}\n")

#     print(f"✓ Saved word list: {txt_path}")

#     # 2. Save as TSV with word column
#     tsv_path = output_dir / f"{output_base}_filtered.tsv"
#     df = pd.DataFrame({'word': sorted(pali_only)})
#     df.to_csv(tsv_path, sep='\t', index=False, encoding='utf-8')

#     print(f"✓ Saved TSV: {tsv_path}")

#     # 3. If original TSV had metadata, preserve it for filtered words
#     if original_tsv and Path(original_tsv).exists():
#         try:
#             original_df = pd.read_csv(original_tsv, sep='\t', encoding='utf-8')

#             # Filter to keep only non-overlapping words
#             if 'word' in original_df.columns:
#                 filtered_df = original_df[original_df['word'].isin(pali_only)]

#                 tsv_full_path = output_dir / f"{output_base}_filtered_full.tsv"
#                 filtered_df.to_csv(tsv_full_path, sep='\t', index=False, encoding='utf-8')

#                 print(f"✓ Saved TSV with metadata: {tsv_full_path}")
#         except Exception as e:
#             print(f"⚠ Could not preserve metadata: {e}")

#     return txt_path, tsv_path


# def save_overlap_report(analysis: Dict, output_path: str):
#     """
#     Save detailed overlap analysis report.
#     """
#     with open(output_path, 'w', encoding='utf-8') as f:
#         f.write("="*80 + "\n")
#         f.write("SINHALA-PALI LEXICON OVERLAP ANALYSIS\n")
#         f.write("="*80 + "\n\n")

#         # Summary statistics
#         f.write("SUMMARY\n")
#         f.write("-"*80 + "\n")
#         f.write(f"Sinhala lexicon size:        {analysis['sinhala_total']:,} words\n")
#         f.write(f"Pali lexicon size:           {analysis['pali_total']:,} words\n")
#         f.write(f"Overlapping words:           {analysis['overlap_count']:,} words\n")
#         f.write(f"Overlap percentage (Pali):   {analysis['overlap_percentage']:.2f}%\n")
#         f.write(f"Sinhala-only words:          {analysis['sinhala_only']:,} words\n")
#         f.write(f"Pali-only words:             {analysis['pali_only']:,} words\n")
#         f.write(f"Total unique words:          {analysis['total_unique_words']:,} words\n\n")

#         # Overlapping words list
#         f.write("="*80 + "\n")
#         f.write("OVERLAPPING WORDS\n")
#         f.write("="*80 + "\n")
#         f.write(f"Total: {len(analysis['overlapping_words'])} words\n\n")

#         for i, word in enumerate(analysis['overlapping_words'], 1):
#             f.write(f"{i:4d}. {word}\n")

#         f.write("\n" + "="*80 + "\n")
#         f.write("END OF REPORT\n")
#         f.write("="*80 + "\n")

#     print(f"✓ Saved overlap report: {output_path}")


# def print_analysis_summary(analysis: Dict):
#     """Print analysis summary to console."""

#     print("\n" + "="*80)
#     print("LEXICON OVERLAP ANALYSIS")
#     print("="*80)

#     print(f"\n📊 LEXICON SIZES:")
#     print(f"  Sinhala lexicon:  {analysis['sinhala_total']:,} words")
#     print(f"  Pali lexicon:     {analysis['pali_total']:,} words")

#     print(f"\n🔍 OVERLAP ANALYSIS:")
#     print(f"  Overlapping words:     {analysis['overlap_count']:,} words")
#     print(f"  Overlap percentage:    {analysis['overlap_percentage']:.2f}% of Pali lexicon")

#     print(f"\n✨ UNIQUE WORDS:")
#     print(f"  Sinhala-only:     {analysis['sinhala_only']:,} words")
#     print(f"  Pali-only:        {analysis['pali_only']:,} words")
#     print(f"  Total unique:     {analysis['total_unique_words']:,} words")

#     # Show sample of overlapping words
#     if analysis['overlapping_words']:
#         print(f"\n📝 SAMPLE OF OVERLAPPING WORDS (first 20):")
#         for word in analysis['overlapping_words'][:20]:
#             print(f"  • {word}")

#         if len(analysis['overlapping_words']) > 20:
#             print(f"  ... and {len(analysis['overlapping_words']) - 20} more")

#     print("\n" + "="*80 + "\n")


# def analyze_overlap_patterns(overlapping_words: list) -> Dict:
#     """
#     Analyze patterns in overlapping words.

#     Returns statistics about word characteristics.
#     """
#     if not overlapping_words:
#         return {}

#     # Length distribution
#     lengths = [len(word) for word in overlapping_words]

#     # Character patterns
#     patterns = {
#         'short_words': sum(1 for w in overlapping_words if len(w) <= 3),
#         'medium_words': sum(1 for w in overlapping_words if 4 <= len(w) <= 7),
#         'long_words': sum(1 for w in overlapping_words if len(w) >= 8),
#         'avg_length': sum(lengths) / len(lengths) if lengths else 0,
#         'min_length': min(lengths) if lengths else 0,
#         'max_length': max(lengths) if lengths else 0
#     }

#     return patterns


# # ============================================
# # MAIN SCRIPT
# # ============================================

# def main():
#     """Main execution function."""

#     # Define paths (adjust to your setup)
#     BASE_DIR = Path(".")
#     SINHALA_LEXICON = BASE_DIR / "data" / "sinhala_lexicon.tsv"
#     PALI_LEXICON = BASE_DIR / "data" / "Pali Lexicon Filtered.tsv"
#     OUTPUT_DIR = BASE_DIR / "data" / "filtered_lexicons"

#     # Create output directory
#     OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#     print("="*80)
#     print("LEXICON OVERLAP DETECTOR & FILTER")
#     print("="*80)

#     # Step 1: Load lexicons
#     print("\n📖 Loading lexicons...")

#     try:
#         sinhala_lex = load_sinhala_lexicon(str(SINHALA_LEXICON))
#         print(f"✓ Loaded Sinhala lexicon: {len(sinhala_lex):,} words")
#     except FileNotFoundError:
#         print(f"✗ Sinhala lexicon not found: {SINHALA_LEXICON}")
#         print("Please download from: https://raw.githubusercontent.com/google/language-resources/master/si/data/lexicon.tsv")
#         return

#     try:
#         pali_lex = load_pali_lexicon(str(PALI_LEXICON))
#         print(f"✓ Loaded Pali lexicon: {len(pali_lex):,} words")
#     except FileNotFoundError:
#         print(f"✗ Pali lexicon not found: {PALI_LEXICON}")
#         return

#     # Step 2: Analyze overlap
#     print("\n🔍 Analyzing overlap...")
#     analysis = analyze_overlap(sinhala_lex, pali_lex)

#     # Step 3: Print summary
#     print_analysis_summary(analysis)

#     # Step 4: Analyze overlap patterns
#     if analysis['overlapping_words']:
#         print("📊 Analyzing overlap patterns...")
#         patterns = analyze_overlap_patterns(analysis['overlapping_words'])

#         print(f"\n  Word length distribution:")
#         print(f"    Short (≤3 chars):  {patterns['short_words']} words")
#         print(f"    Medium (4-7):      {patterns['medium_words']} words")
#         print(f"    Long (≥8):         {patterns['long_words']} words")
#         print(f"    Average length:    {patterns['avg_length']:.1f} characters")

#     # Step 5: Save filtered Pali lexicon
#     print("\n💾 Saving filtered Pali lexicon...")

#     filtered_base = OUTPUT_DIR / "pali_lexicon_no_overlap"
#     txt_path, tsv_path = save_filtered_pali_lexicon(
#         set(analysis['pali_unique_words']),
#         str(filtered_base),
#         original_tsv=str(PALI_LEXICON)
#     )

#     # Step 6: Save overlap report
#     report_path = OUTPUT_DIR / "overlap_analysis_report.txt"
#     save_overlap_report(analysis, str(report_path))

#     # Step 7: Save overlapping words list
#     overlap_path = OUTPUT_DIR / "overlapping_words.txt"
#     with open(overlap_path, 'w', encoding='utf-8') as f:
#         f.write("Overlapping Words Between Sinhala and Pali Lexicons\n")
#         f.write("="*80 + "\n\n")
#         for i, word in enumerate(analysis['overlapping_words'], 1):
#             f.write(f"{i:4d}. {word}\n")

#     print(f"✓ Saved overlapping words: {overlap_path}")

#     # Final summary
#     print("\n" + "="*80)
#     print("✅ PROCESSING COMPLETE")
#     print("="*80)
#     print(f"\nGenerated files:")
#     print(f"  1. Filtered Pali lexicon (TXT):  {txt_path}")
#     print(f"  2. Filtered Pali lexicon (TSV):  {tsv_path}")
#     print(f"  3. Overlap analysis report:      {report_path}")
#     print(f"  4. Overlapping words list:       {overlap_path}")

#     print(f"\n📈 RESULTS:")
#     print(f"  Original Pali lexicon:  {analysis['pali_total']:,} words")
#     print(f"  Removed (overlap):      {analysis['overlap_count']:,} words ({analysis['overlap_percentage']:.2f}%)")
#     print(f"  Filtered Pali lexicon:  {len(analysis['pali_unique_words']):,} words")
#     print(f"  Reduction:              {analysis['overlap_count']:,} fewer words")

#     print("\n" + "="*80 + "\n")


# if __name__ == "__main__":
#     main()

In [ ]:
# def download_google_lexicon(save_path: str = "lexicon.tsv") -> str:
#     """
#     Download the full Google Sinhala lexicon from GitHub.
#     """
#     url = "https://raw.githubusercontent.com/google/language-resources/master/si/data/lexicon.tsv"

#     print(f"Downloading Sinhala lexicon from {url}...")

#     try:
#         urllib.request.urlretrieve(url, save_path)
#         print(f"✓ Lexicon downloaded to: {save_path}")
#         return save_path
#     except Exception as e:
#         print(f"✗ Download failed: {e}")
#         print("Please download manually from:")
#         print(url)
#         raise

In [ ]:
# import urllib
# download_google_lexicon("data/sinhala_lexicon.tsv")